In [31]:
import os
import joblib
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.base import ClassifierMixin, RegressorMixin


def load_pipeline_model(model_path: str) -> Pipeline:
    """
    Charge une Pipeline sklearn depuis un fichier .joblib dont le nom
    encode le label modélisé. Vérifie que l'objet chargé est bien
    une sklearn.pipeline.Pipeline et signale la présence d'un pas XGBRegressor.
    
    Parameters
    ----------
    model_path : str
        Chemin vers le fichier .joblib (ex. '.../xgbregressor_normalized_ENB2012_data.csv_l_heating_load.joblib')
    
    Returns
    -------
    pipeline : sklearn.pipeline.Pipeline
        La pipeline chargée.
    
    Raises
    ------
    ValueError
        Si le nom de fichier n'est pas au format attendu.
    TypeError
        Si l'objet chargé n'est pas une Pipeline.
    """
    # 1. Extraction du label depuis le nom de fichier
    basename    = os.path.basename(model_path)
    if not basename.endswith('.joblib'):
        raise ValueError(f"Le fichier '{basename}' doit se terminer par '.joblib'")
    
    name_no_ext = basename[:-7]  # enlève '.joblib'
    try:
        # on ignore la première partie (classe) avant le premier '_'
        _, rest  = name_no_ext.split('_', 1)
        _, label = rest.split('.csv_', 1)
    except ValueError:
        raise ValueError(
            f"Le nom '{basename}' n'est pas au format attendu "
            "(<classe>_<nom.csv>_<label>.joblib)"
        )
    
    # 2. Chargement
    pipeline = joblib.load(model_path)
    
    # 3. Vérification du type
    if not isinstance(pipeline, Pipeline):
        raise TypeError(
            f"Objet chargé de type {type(pipeline).__name__} inattendu, "
            "attendu sklearn.pipeline.Pipeline"
        )
    
    # 4. inspection des estimateurs
    found = []
    for name, step in pipeline.named_steps.items():
        if isinstance(step, (ClassifierMixin, RegressorMixin)):
            found.append((name, step.__class__.__name__))

    if found:
        print("✅ Estimateurs détectés dans la pipeline :")
        for step_name, cls_name in found:
            print(f"  - {step_name}: {cls_name}")
    else:
        print("⚠️ Aucune étape de modélisation scikit-learn trouvée.")

    # 5. On retourne simplement la Pipeline
    print(f"Pipeline chargée pour le label '{label}'.")
    return pipeline

In [33]:
model_path_l_heating_load = 'saved_models/xgbregressor_normalized_ENB2012_data.csv_l_heating_load.joblib'
model_path_l_cooling_load = 'saved_models/lgbmregressor_normalized_ENB2012_data.csv_l_cooling_load.joblib'


pipeline_l_heating_load = load_pipeline_model(model_path_l_heating_load)
pipeline_l_cooling_load = load_pipeline_model(model_path_l_cooling_load)


✅ Estimateurs détectés dans la pipeline :
  - model: XGBRegressor
Pipeline chargée pour le label 'l_heating_load'.
✅ Estimateurs détectés dans la pipeline :
  - model: LGBMRegressor
Pipeline chargée pour le label 'l_cooling_load'.


## 